# CDC in the Medallion Architecture (Change Data Capture)

**Topics covered in this module:**
1. What is **CDC (Change Data Capture)** and why is it used?
2. What a **CDC feed** looks like (INSERT / UPDATE / DELETE)
3. **Bronze**: land the change feed exactly as received
4. **Silver**: apply the changes to reflect the *current state* (using `MERGE INTO`)
5. **Gold**: build business aggregations on top of the current state
6. The production-ready declarative approach: **`AUTO CDC INTO`** (formerly `APPLY CHANGES INTO`)

## Important Note: Free Edition / Community Edition (Serverless)

Two important notes for this CDC module:

### How We Simulate the CDC Feed

In production, a CDC feed is generated by a tool such as Debezium or AWS DMS, which reads the *transaction log* of the source database. In this module, **we do not set up that infrastructure**. Instead, we simulate the feed by **dropping JSON files into a Volume**. Each JSON line represents a change event and includes an `operation` column (`INSERT` / `UPDATE` / `DELETE`) along with a `sequence_num` column that defines the event order.

### Why We Use `MERGE INTO` Instead of `AUTO CDC INTO`

In Databricks, the **production-ready** way to apply a CDC feed is with the declarative **`AUTO CDC INTO`** command (which replaced `APPLY CHANGES INTO`). However, this command **only works inside a Lakeflow Declarative Pipelines pipeline**, not in a standalone notebook cell.

To allow you to execute the notebook **step by step** and clearly understand how CDC works, we manually apply the changes in the Silver layer using **`MERGE INTO`**. This is **the same underlying logic that `AUTO CDC INTO` performs automatically** (upserts, deletes, and sequence-based ordering). At the end of the notebook, we also show the equivalent `AUTO CDC INTO` implementation so you can become familiar with the official syntax.

> **In summary:** **`MERGE INTO`** is the manual implementation of the CDC logic you learn in this module. **`AUTO CDC INTO`** performs the same logic automatically within a production Lakeflow Declarative Pipelines pipeline.

---
## 0. Setup

We follow the same setup pattern as in the previous modules: create the catalog, schema, and volume, along with a `landing/` directory where the simulated CDC events are dropped.


In [0]:
# Environment variables
catalog = "main"
schema  = "medallion_cdc"
volume  = "raw_data"

base_path = f"/Volumes/{catalog}/{schema}/{volume}"
landing   = f"{base_path}/landing"          # where CDC feed events are dropped

print("Base path :", base_path)
print("Landing   :", landing)

Base path : /Volumes/main/medallion_cdc/raw_data
Landing   : /Volumes/main/medallion_cdc/raw_data/landing


In [0]:
# Create the catalog / schema / volume structure if it does not exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {catalog}.{schema}.{volume}")

# Set the default schema so %sql can be used without prefixes
spark.sql(f"USE {catalog}.{schema}")
print(f"Environment ready. Using {catalog}.{schema}")

Environment ready. Using main.medallion_cdc


In [0]:
# Helper: write a text file to the Volume
# In Serverless, we can write directly to /Volumes/... using Python's open().
# We'll use this helper to "drop" files into the landing directory and simulate
# the arrival of a CDC feed.
import os

def drop_file(folder, filename, content):
    os.makedirs(folder, exist_ok=True)
    path = f"{folder}/{filename}"
    with open(path, "w") as f:
        f.write(content)
    print(f"  + file created: {path}")

print("drop_file() helper is ready")

drop_file() helper is ready


---
## 1. What Is CDC (Change Data Capture)?

CDC is a **design pattern** that captures **changes** made to a data source—**inserts, updates, and deletes**—and propagates them to another system, instead of copying the entire table every time.

**Why does it exist?** Imagine an `orders` database containing millions of rows. Copying the entire table into your lakehouse every night would be slow and expensive. With CDC, you only transfer **what has changed** since the last synchronization.

| Concept | Description |
|---|---|
| **CDC Source** | The source database being monitored (e.g., PostgreSQL, MySQL) |
| **CDC Feed** | The stream of change events emitted by the source (INSERT / UPDATE / DELETE) |
| **Target** | The destination table that stays synchronized with the source |

**The core challenge of CDC:** the feed is a **sequence of operations**, not the current state. If an order is created and later updated, the feed contains *two* events for that order. The job of the Silver layer is to **collapse those events** and keep only the **latest state** of each record. That's exactly what we'll learn in this module.

---
## 2. The CDC Feed — First Batch of Changes

We'll simulate the CDC feed for an `orders` table. Each line represents a **change event** containing:

- The order data (`order_id`, `customer`, `product`, `amount`, `status`)
- `operation`: the type of change (`INSERT`, `UPDATE`, `DELETE`)
- `sequence_num`: the **order** in which the change occurred (essential for applying events correctly)

**First batch:** three new orders arrive (all `INSERT` events).


In [0]:
# Drop the FIRST CDC feed batch (3 inserts)
# JSON Lines format: one change event per line.
batch_1 = "\n".join([
    '{"order_id": 1, "customer": "Ana",   "product": "Sneakers", "amount": 90.0,  "status": "pending", "operation": "INSERT", "sequence_num": 1}',
    '{"order_id": 2, "customer": "Beto",  "product": "T-Shirt",  "amount": 25.0,  "status": "pending", "operation": "INSERT", "sequence_num": 2}',
    '{"order_id": 3, "customer": "Carla", "product": "Cap",      "amount": 15.0,  "status": "pending", "operation": "INSERT", "sequence_num": 3}',
])

drop_file(landing, "cdc_batch_01.json", batch_1)
print("\nFirst CDC feed batch has been dropped into the landing directory (3 INSERT events).")

  + file created: /Volumes/main/medallion_cdc/raw_data/landing/cdc_batch_01.json

First CDC feed batch has been dropped into the landing directory (3 INSERT events).


---
## 3. Bronze Layer (Landing the CDC Feed As-Is)

The Bronze layer is a **faithful copy** of the CDC feed. At this stage, **nothing is merged or collapsed**—every change event is stored exactly as it arrives, including the `operation` and `sequence_num` columns.

The Bronze layer is **append-only**: each new CDC batch is simply **added** to the historical log of events. This serves as the **source of truth**. If anything goes wrong downstream, you still have the complete history of changes available for replay and reprocessing.

In [0]:
# BRONZE: Ingest the CDC feed events WITHOUT transforming them
from pyspark.sql.functions import current_timestamp, col

def ingest_to_bronze():
    """Reads all files from the landing directory and appends new events to bronze_orders_cdc."""
    raw_df = (
        spark.read
          .option("multiLine", "false")   # JSON Lines: one JSON object per line
          .json(landing)
    )

    bronze_df = (
        raw_df
          .withColumn("_ingested_at", current_timestamp())
          .withColumn("_source_file", col("_metadata.file_path"))
    )

    # Append mode: Bronze stores the COMPLETE history of change events
    (
        bronze_df.write
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable("bronze_orders_cdc")
    )

    print("Events appended to bronze_orders_cdc (append mode).")

ingest_to_bronze()

Events appended to bronze_orders_cdc (append mode).


In [0]:
%sql
-- Bronze: the raw list of change events
-- Notice that operation and sequence_num are preserved.
-- Bronze does NOT merge or collapse events.
SELECT order_id, customer, product, amount, status, operation, sequence_num
FROM   bronze_orders_cdc
ORDER  BY sequence_num

order_id,customer,product,amount,status,operation,sequence_num
1,Ana,Sneakers,90.0,pending,INSERT,1
2,Beto,T-Shirt,25.0,pending,INSERT,2
3,Carla,Cap,15.0,pending,INSERT,3


---
## 4. Silver Layer: Applying the Changes (The Core of CDC)

The Silver layer should always represent the **current state** of each order: **one row per `order_id`**, containing its latest version. Since the first batch contains only three `INSERT` events, the Silver table will simply contain those three orders.

We'll create the Silver table and apply the CDC feed using **`MERGE INTO`**, which performs the upsert logic:

- If the `order_id` **already exists** → **UPDATE** the existing row.
- If the `order_id` **does not exist** → **INSERT** a new row.
- If the event is a **DELETE** → remove the row.

First, we'll create an empty Silver table using the final-state schema (excluding `operation` and `sequence_num`, which are metadata from the CDC feed).


In [0]:
%sql
-- Create the Silver table (current status, one row per order)
CREATE TABLE IF NOT EXISTS silver_orders (
  order_id  INT,
  customer  STRING,
  product   STRING,
  amount    DOUBLE,
  status    STRING
)

### The Key Step: Deduplicate the CDC Feed Before the MERGE

A `MERGE` statement cannot process **multiple events for the same `order_id` in a single operation**—doing so would result in an error. In a real CDC feed, however, the same order may appear multiple times within the same batch.

To handle this correctly, we keep **only the latest event for each `order_id`**, based on the `sequence_num`. The following function encapsulates this core CDC pattern and will be reused for every new batch.


In [0]:
# SILVER: Apply the CDC feed using MERGE INTO (upsert + delete)
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

def apply_cdc_to_silver():
    """
    Manually reproduces what AUTO CDC INTO does automatically:
    1) Orders the CDC feed by sequence_num
    2) Keeps only the LATEST event for each order_id
    3) MERGE: deletes rows for DELETE events, otherwise performs an upsert
    """
    bronze = spark.table("bronze_orders_cdc")

    # 1 + 2) Keep the latest event for each order_id based on the sequence number
    w = Window.partitionBy("order_id").orderBy(col("sequence_num").desc())
    latest_events = (
        bronze
            .withColumn("_rn", row_number().over(w))
            .filter(col("_rn") == 1)      # keep only the most recent change per order
            .drop("_rn")
    )

    latest_events.createOrReplaceTempView("cdc_latest")

    # 3) MERGE: apply the latest state to the Silver table
    spark.sql("""
        MERGE INTO silver_orders AS t
        USING cdc_latest AS s
          ON t.order_id = s.order_id
        WHEN MATCHED AND s.operation = 'DELETE' THEN DELETE
        WHEN MATCHED AND s.operation != 'DELETE' THEN UPDATE SET
            t.customer = s.customer,
            t.product  = s.product,
            t.amount   = s.amount,
            t.status   = s.status
        WHEN NOT MATCHED AND s.operation != 'DELETE' THEN INSERT
            (order_id, customer, product, amount, status)
            VALUES (s.order_id, s.customer, s.product, s.amount, s.status)
    """)

    print("CDC successfully applied to silver_orders.")

apply_cdc_to_silver()

CDC successfully applied to silver_orders.


In [0]:
%sql
-- Silver: current status after the first batch (3 orders, all pending)
SELECT order_id, customer, product, amount, status
FROM   silver_orders
ORDER  BY order_id

order_id,customer,product,amount,status
1,Ana,Sneakers,90.0,pending
2,Beto,T-Shirt,25.0,pending
3,Carla,Cap,15.0,pending


---
## 5. A SECOND Batch Arrives: UPDATE + DELETE + INSERT

This is where CDC becomes interesting. A new batch of change events arrives for orders that **already exist**:

- **Order 1** (Ana) → its status changes to `shipped` (**UPDATE**)
- **Order 2** (Beto) → the order is canceled, so it is removed (**DELETE**)
- **Order 4** (Diana) → a brand-new order (**INSERT**)

Notice that the `sequence_num` continues increasing (4, 5, 6). This allows the pipeline to determine that these events occurred **after** those in the first batch.


In [0]:
# Drop the SECOND CDC feed batch (UPDATE + DELETE + INSERT)
batch_2 = "\n".join([
    '{"order_id": 1, "customer": "Ana",   "product": "Sneakers", "amount": 90.0, "status": "shipped",  "operation": "UPDATE", "sequence_num": 4}',
    '{"order_id": 2, "customer": "Beto",  "product": "T-Shirt",  "amount": 25.0, "status": "canceled", "operation": "DELETE", "sequence_num": 5}',
    '{"order_id": 4, "customer": "Diana", "product": "Hoodie",   "amount": 60.0, "status": "pending",  "operation": "INSERT", "sequence_num": 6}',
])

drop_file(landing, "cdc_batch_02.json", batch_2)
print("\nSecond CDC feed batch has been dropped into the landing directory (1 UPDATE, 1 DELETE, 1 INSERT event).")

  + file created: /Volumes/main/medallion_cdc/raw_data/landing/cdc_batch_02.json

Second CDC feed batch has been dropped into the landing directory (1 UPDATE, 1 DELETE, 1 INSERT event).


Now we'll **re-ingest the Bronze layer** (which reads the entire landing directory again, now containing both batches) and **reapply the CDC logic to the Silver layer**. Since Bronze stores the complete history of change events and the `MERGE` keeps only the latest event for each order, the Silver table will be fully synchronized with the current state.


In [0]:
# Reprocess: Bronze (entire CDC feed) -> Silver (current state)

# Recreate the Bronze table from the entire landing directory
# (which now contains both batch 1 and batch 2)
spark.sql("DROP TABLE IF EXISTS bronze_orders_cdc")
ingest_to_bronze()

# Reapply the CDC feed to the Silver table
apply_cdc_to_silver()

Events appended to bronze_orders_cdc (append mode).
CDC successfully applied to silver_orders.


In [0]:
%sql
-- Bronze now contains all 6 events (the complete history of the CDC feed)
SELECT order_id, status, operation, sequence_num
FROM   bronze_orders_cdc
ORDER  BY sequence_num

order_id,status,operation,sequence_num
1,pending,INSERT,1
2,pending,INSERT,2
3,pending,INSERT,3
1,shipped,UPDATE,4
2,canceled,DELETE,5
4,pending,INSERT,6


In [0]:
%sql
-- Silver: the CURRENT STATE after applying all CDC events
-- Order 1: now 'shipped' (updated)
-- Order 2: NO LONGER EXISTS (removed by the DELETE event)
-- Order 3: unchanged
-- Order 4: newly inserted (Diana)
SELECT order_id, customer, product, amount, status
FROM   silver_orders
ORDER  BY order_id

order_id,customer,product,amount,status
1,Ana,Sneakers,90.0,shipped
3,Carla,Cap,15.0,pending
4,Diana,Hoodie,60.0,pending


---
## 6. Gold Layer — Business Aggregations

The Gold layer no longer operates at the individual row level. Instead, it aggregates the **current state** stored in the Silver layer into **business metrics**. In this example, we'll calculate total sales and the number of active orders by status.

Because Gold is built on top of Silver's **already consolidated state**, the deleted Order 2 is **no longer included** in the metrics—which is exactly the expected behavior.


In [0]:
%sql
-- GOLD: Business metrics based on the current state
CREATE OR REPLACE TABLE gold_order_summary AS
SELECT
  status,
  COUNT(*) AS num_orders,
  ROUND(SUM(amount), 2) AS total_amount
FROM   silver_orders
GROUP BY status
ORDER BY status

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Gold: the final business table ready for dashboards
SELECT *
FROM gold_order_summary
ORDER BY status

status,num_orders,total_amount
pending,2,75.0
shipped,1,90.0


---
## 7. The Production Approach: `AUTO CDC INTO`

Everything we manually implemented in the Silver layer using `MERGE INTO` plus window-based deduplication is **exactly what Databricks automates** with `AUTO CDC INTO` (formerly `APPLY CHANGES INTO`). In a **Lakeflow Declarative Pipelines** pipeline, the entire Silver layer could be reduced to the following:

```sql
-- (This code belongs INSIDE a Lakeflow pipeline, not in a notebook cell)

-- 1) The target table
CREATE OR REFRESH STREAMING TABLE silver_orders;

-- 2) The CDC flow: replaces all the MERGE + Window + deduplication logic
CREATE FLOW silver_orders_flow AS AUTO CDC INTO silver_orders
FROM STREAM(bronze_orders_cdc)
KEYS (order_id)                              -- Primary key (equivalent to ON t.order_id = s.order_id)
APPLY AS DELETE WHEN operation = "DELETE"    -- Delete logic (equivalent to WHEN MATCHED ... THEN DELETE)
SEQUENCE BY sequence_num                     -- Event ordering (equivalent to Window + row_number)
COLUMNS * EXCEPT (operation, sequence_num);  -- Columns to keep (exclude CDC metadata)
```

Compare it line by line with the manual implementation:

| Manual Implementation | `AUTO CDC INTO` Clause |
|---|---|
| `MERGE ... ON t.order_id = s.order_id` | `KEYS (order_id)` |
| `WHEN MATCHED AND operation='DELETE' THEN DELETE` | `APPLY AS DELETE WHEN operation = "DELETE"` |
| `Window.partitionBy(...).orderBy(sequence_num.desc())` + `row_number()` | `SEQUENCE BY sequence_num` |
| `INSERT / UPDATE SET ...` (upsert) | Default behavior of `AUTO CDC` |
| `.drop("operation", "sequence_num")` | `COLUMNS * EXCEPT (operation, sequence_num)` |

**Why use `AUTO CDC INTO`?** It reduces all of that manual logic to just a few lines of declarative code, automatically handles late or out-of-order events, and even supports **full historical tracking** with SCD Type 2 by adding `STORED AS SCD TYPE 2`.

**Why didn't we run it in this module?** Because it requires a separate Lakeflow Declarative Pipelines pipeline (with its own execution environment and UI), which would interrupt the step-by-step notebook experience. The important part is that you now understand **exactly what `AUTO CDC INTO` does under the hood.** 🎯

---
## Clean Up


In [0]:
def clean_up():
    print("Dropping tables...")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.bronze_orders_cdc")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.silver_orders")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.gold_order_summary")

    print("Deleting files from the volume...")
    dbutils.fs.rm(landing, True)

    print("Dropping schema...")
    spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE")

    print("Done ✓")

In [0]:
# clean_up()
